# Parameter counts for every model in Section 5

Re-creates each encoder class with the exact constructor args used at training time and counts `sum(p.numel() for p in m.parameters())`. The decoder is shared across every GNN model: a 2-layer MLP `Linear(2*emb_dim -> 128) -> ReLU -> Dropout -> Linear(128 -> 1)`. The MLP-only baseline uses the same decoder shape but on raw 768-dim features.

Use the project venv (`torch_geometric` installed). Run all cells; the last cell prints a paste-ready table.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GATConv, GATv2Conv

INPUT_DIM = 768
HIDDEN_DIM = 256
OUTPUT_DIM_GAT = 256       # GAT / HetGAT / RGAT / GATv2
OUTPUT_DIM_GRAPHSAGE = 128  # GraphSAGE notebook used 128 out
DECODER_HIDDEN_DIM = 128
RELATIONS = ['cites', 'cited_by']

def n_params(m):
    return sum(p.numel() for p in m.parameters())

In [ ]:
# ---- Decoder (shared by every GNN model) ----
class MLPDecoder(nn.Module):
    def __init__(self, emb_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(2 * emb_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

# ---- HAN semantic-level attention (used by HetGAT-* only) ----
class SemanticAttention(nn.Module):
    def __init__(self, in_dim, attn_dim=128):
        super().__init__()
        self.proj = nn.Linear(in_dim, attn_dim)
        self.q = nn.Parameter(torch.empty(attn_dim))
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
        nn.init.normal_(self.q, mean=0.0, std=0.1)

In [ ]:
# ---- Homogeneous encoders ----
class GraphSAGEEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, output_dim)
        self.dropout = dropout

class GATEncoder(nn.Module):  # 2L
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads_l1=4, heads_l2=1, dropout=0.2,
                 attn_dropout=0.0, negative_slope=0.2):
        super().__init__()
        per = hidden_dim // heads_l1
        self.conv1 = GATConv(input_dim, per, heads=heads_l1, concat=True,
                             negative_slope=negative_slope, dropout=attn_dropout)
        self.conv2 = GATConv(hidden_dim, output_dim, heads=heads_l2,
                             concat=(heads_l2 > 1),
                             negative_slope=negative_slope, dropout=attn_dropout)

class GAT3LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads_l1=4, middle_heads=4, heads_l3=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = GATConv(input_dim, per1, heads=heads_l1, concat=True)
        self.conv2 = GATConv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv3 = GATConv(hidden_dim, output_dim, heads=heads_l3,
                             concat=(heads_l3 > 1))

class GAT4LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads_l1=4, middle_heads=4, heads_l4=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = GATConv(input_dim, per1, heads=heads_l1, concat=True)
        self.conv2 = GATConv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv3 = GATConv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv4 = GATConv(hidden_dim, output_dim, heads=heads_l4,
                             concat=(heads_l4 > 1))

class GAT5LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads_l1=4, middle_heads=4, heads_l5=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = GATConv(input_dim, per1, heads=heads_l1, concat=True)
        self.conv2 = GATConv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv3 = GATConv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv4 = GATConv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv5 = GATConv(hidden_dim, output_dim, heads=heads_l5,
                             concat=(heads_l5 > 1))

class GATv24LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 heads_l1=4, middle_heads=4, heads_l4=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = GATv2Conv(input_dim, per1, heads=heads_l1, concat=True)
        self.conv2 = GATv2Conv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv3 = GATv2Conv(hidden_dim, perm, heads=middle_heads, concat=True)
        self.conv4 = GATv2Conv(hidden_dim, output_dim, heads=heads_l4,
                               concat=(heads_l4 > 1))

In [ ]:
# ---- Heterogeneous (per-relation) encoders ----
def _per_rel(in_d, out_d, heads, concat, relations):
    return nn.ModuleDict({
        rel: GATConv(in_d, out_d, heads=heads, concat=concat)
        for rel in relations
    })

class HeteroGATEncoder(nn.Module):  # 2L (HAN)
    def __init__(self, input_dim, hidden_dim, output_dim, relations,
                 heads_l1=4, heads_l2=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        self.conv1 = _per_rel(input_dim, per1, heads_l1, True, relations)
        self.attn1 = SemanticAttention(hidden_dim)
        self.conv2 = _per_rel(hidden_dim, output_dim, heads_l2,
                              (heads_l2 > 1), relations)
        self.attn2 = SemanticAttention(output_dim)

class HeteroGAT3LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, relations,
                 heads_l1=4, middle_heads=4, heads_l3=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = _per_rel(input_dim, per1, heads_l1, True, relations)
        self.attn1 = SemanticAttention(hidden_dim)
        self.conv2 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.attn2 = SemanticAttention(hidden_dim)
        self.conv3 = _per_rel(hidden_dim, output_dim, heads_l3,
                              (heads_l3 > 1), relations)
        self.attn3 = SemanticAttention(output_dim)

class HeteroGAT4LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, relations,
                 heads_l1=4, middle_heads=4, heads_l4=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = _per_rel(input_dim, per1, heads_l1, True, relations)
        self.attn1 = SemanticAttention(hidden_dim)
        self.conv2 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.attn2 = SemanticAttention(hidden_dim)
        self.conv3 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.attn3 = SemanticAttention(hidden_dim)
        self.conv4 = _per_rel(hidden_dim, output_dim, heads_l4,
                              (heads_l4 > 1), relations)
        self.attn4 = SemanticAttention(output_dim)

class HeteroGAT5LEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, relations,
                 heads_l1=4, middle_heads=4, heads_l5=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = _per_rel(input_dim, per1, heads_l1, True, relations)
        self.attn1 = SemanticAttention(hidden_dim)
        self.conv2 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.attn2 = SemanticAttention(hidden_dim)
        self.conv3 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.attn3 = SemanticAttention(hidden_dim)
        self.conv4 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.attn4 = SemanticAttention(hidden_dim)
        self.conv5 = _per_rel(hidden_dim, output_dim, heads_l5,
                              (heads_l5 > 1), relations)
        self.attn5 = SemanticAttention(output_dim)

class RGAT3LEncoder(nn.Module):  # per-relation GATs, no SemanticAttention
    def __init__(self, input_dim, hidden_dim, output_dim, relations,
                 heads_l1=4, middle_heads=4, heads_l3=1, **kw):
        super().__init__()
        per1 = hidden_dim // heads_l1
        perm = hidden_dim // middle_heads
        self.conv1 = _per_rel(input_dim, per1, heads_l1, True, relations)
        self.conv2 = _per_rel(hidden_dim, perm, middle_heads, True, relations)
        self.conv3 = _per_rel(hidden_dim, output_dim, heads_l3,
                              (heads_l3 > 1), relations)

In [ ]:
# ---- Build, count, print ----
decoder_gat   = MLPDecoder(OUTPUT_DIM_GAT,       DECODER_HIDDEN_DIM)
decoder_sage  = MLPDecoder(OUTPUT_DIM_GRAPHSAGE, DECODER_HIDDEN_DIM)
decoder_mlp   = MLPDecoder(INPUT_DIM,            DECODER_HIDDEN_DIM)

rows = []  # (name, encoder_params, decoder_params)
rows.append(("MLP-only",     0, n_params(decoder_mlp)))
rows.append(("GraphSAGE-2L", n_params(GraphSAGEEncoder(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM_GRAPHSAGE)), n_params(decoder_sage)))
for L, cls in [(2, GATEncoder), (3, GAT3LEncoder), (4, GAT4LEncoder), (5, GAT5LEncoder)]:
    rows.append((f"GAT-{L}L", n_params(cls(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM_GAT)), n_params(decoder_gat)))
rows.append(("GATv2-4L", n_params(GATv24LEncoder(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM_GAT)), n_params(decoder_gat)))
for L, cls in [(2, HeteroGATEncoder), (3, HeteroGAT3LEncoder), (4, HeteroGAT4LEncoder), (5, HeteroGAT5LEncoder)]:
    rows.append((f"HetGAT-{L}L", n_params(cls(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM_GAT, RELATIONS)), n_params(decoder_gat)))
rows.append(("R-GAT-3L", n_params(RGAT3LEncoder(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM_GAT, RELATIONS)), n_params(decoder_gat)))

print(f"{'Model':<14}  {'Encoder':>10}  {'Decoder':>10}  {'Total':>10}")
print('-' * 50)
for name, enc, dec in rows:
    print(f"{name:<14}  {enc:>10,}  {dec:>10,}  {(enc+dec):>10,}")